In [9]:
import pandas as pd
from glob import glob
import os

# Locate all raw CSV files using a glob pattern.
# Sorting ensures a consistent file ordering across different runs.
files = sorted(glob("../data/raw/*.csv"))

print("Number of files:", len(files))
print()

# Print a quick preview of each file to confirm headers and structure
# before attempting any merge or transformation.
for f in files:
    print("="*80)
    print("FILE:", os.path.basename(f))
    try:
        df = pd.read_csv(f, nrows=5)
        print("Shape preview:", df.shape)
        print("Number of columns:", len(df.columns))
        print("First 5 columns:", list(df.columns[:5]))
        print("Last column:", df.columns[-1])
    except Exception as e:
        print("ERROR:", e)


Number of files: 8

FILE: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Shape preview: (5, 79)
Number of columns: 79
First 5 columns: [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets']
Last column:  Label
FILE: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Shape preview: (5, 79)
Number of columns: 79
First 5 columns: [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets']
Last column:  Label
FILE: Friday-WorkingHours-Morning.pcap_ISCX.csv
Shape preview: (5, 79)
Number of columns: 79
First 5 columns: [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets']
Last column:  Label
FILE: Monday-WorkingHours.pcap_ISCX.csv
Shape preview: (5, 79)
Number of columns: 79
First 5 columns: [' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd 

In [10]:
schema_summary = []

# Build a schema summary for every raw file.
# This verifies that all files share the same column names and order,
# which is a prerequisite for a safe pd.concat merge in the ETL step.
for f in files:
    try:
        cols = pd.read_csv(f, nrows=0).columns.tolist()
        schema_summary.append({
            "file": os.path.basename(f),
            "num_columns": len(cols),
            "first_column": cols[0] if len(cols) > 0 else None,
            "last_column": cols[-1] if len(cols) > 0 else None
        })
    except Exception as e:
        schema_summary.append({
            "file": os.path.basename(f),
            "num_columns": None,
            "first_column": None,
            "last_column": f"ERROR: {e}"
        })

schema_df = pd.DataFrame(schema_summary)
schema_df


,file,num_columns,first_column,last_column
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,79,Destination Port,Label
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,79,Destination Port,Label
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,79,Destination Port,Label
3,Monday-WorkingHours.pcap_ISCX.csv,79,Destination Port,Label
4,Thursday-WorkingHours-Afternoon-Infilteration....,79,Destination Port,Label
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,79,Destination Port,Label
6,Tuesday-WorkingHours.pcap_ISCX.csv,79,Destination Port,Label
7,Wednesday-workingHours.pcap_ISCX.csv,79,Destination Port,Label


In [11]:
all_columns = []

# Collect the exact column list from every file using nrows=0
# (reads only the header row, so memory usage is negligible).
for f in files:
    cols = pd.read_csv(f, nrows=0).columns.tolist()
    all_columns.append(cols)

# Confirm schema consistency: all files must have the same columns
# in the same order so that a vertical stack (concat) is valid.
same_schema = all(cols == all_columns[0] for cols in all_columns)

print("Do all raw files have the same column order and names?", same_schema)
print("Number of columns in first file:", len(all_columns[0]))


Do all raw files have the same column order and names? True
Number of columns in first file: 79


In [12]:
# Load the first file as a representative sample.
# Reading one complete file is sufficient to inspect data types,
# value ranges, and label distribution without loading all data.
sample_file = files[0]
sample_df = pd.read_csv(sample_file)

print("Sample file:", os.path.basename(sample_file))
print("Shape:", sample_df.shape)
print()
print(sample_df.head())


Sample file: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Shape: (225745, 79)

    Destination Port   Flow Duration   Total Fwd Packets  \
0              54865               3                   2   
1              55054             109                   1   
2              55055              52                   1   
3              46236              34                   1   
4              54863               3                   2   

    Total Backward Packets  Total Length of Fwd Packets  \
0                        0                           12   
1                        1                            6   
2                        1                            6   
3                        1                            6   
4                        0                           12   

    Total Length of Bwd Packets   Fwd Packet Length Max  \
0                             0                       6   
1                             6                       6   
2                       

In [13]:
# Print column names using repr() to expose hidden characters
# such as leading/trailing whitespace.
# The CIC-IDS-2017 dataset is known to have a leading space in ' Label',
# which must be stripped during ETL to avoid key errors.
print("Column names from sample file (repr to detect hidden spaces):")
for col in sample_df.columns:
    print(repr(col))


Column names from sample file:
' Destination Port'
' Flow Duration'
' Total Fwd Packets'
' Total Backward Packets'
'Total Length of Fwd Packets'
' Total Length of Bwd Packets'
' Fwd Packet Length Max'
' Fwd Packet Length Min'
' Fwd Packet Length Mean'
' Fwd Packet Length Std'
'Bwd Packet Length Max'
' Bwd Packet Length Min'
' Bwd Packet Length Mean'
' Bwd Packet Length Std'
'Flow Bytes/s'
' Flow Packets/s'
' Flow IAT Mean'
' Flow IAT Std'
' Flow IAT Max'
' Flow IAT Min'
'Fwd IAT Total'
' Fwd IAT Mean'
' Fwd IAT Std'
' Fwd IAT Max'
' Fwd IAT Min'
'Bwd IAT Total'
' Bwd IAT Mean'
' Bwd IAT Std'
' Bwd IAT Max'
' Bwd IAT Min'
'Fwd PSH Flags'
' Bwd PSH Flags'
' Fwd URG Flags'
' Bwd URG Flags'
' Fwd Header Length'
' Bwd Header Length'
'Fwd Packets/s'
' Bwd Packets/s'
' Min Packet Length'
' Max Packet Length'
' Packet Length Mean'
' Packet Length Std'
' Packet Length Variance'
'FIN Flag Count'
' SYN Flag Count'
' RST Flag Count'
' PSH Flag Count'
' ACK Flag Count'
' URG Flag Count'
' CWE Flag 

In [14]:
# Check the label distribution in the sample file.
# Handles both ' Label' (with a leading space, as found in raw files)
# and 'Label' (after stripping) to remain robust across files.
if " Label" in sample_df.columns:
    print("Label distribution in raw sample file:")
    print(sample_df[" Label"].value_counts())
elif "Label" in sample_df.columns:
    print("Label distribution in raw sample file:")
    print(sample_df["Label"].value_counts())
else:
    print("Label column not found.")


Column names from sample file:
' Destination Port'
' Flow Duration'
' Total Fwd Packets'
' Total Backward Packets'
'Total Length of Fwd Packets'
' Total Length of Bwd Packets'
' Fwd Packet Length Max'
' Fwd Packet Length Min'
' Fwd Packet Length Mean'
' Fwd Packet Length Std'
'Bwd Packet Length Max'
' Bwd Packet Length Min'
' Bwd Packet Length Mean'
' Bwd Packet Length Std'
'Flow Bytes/s'
' Flow Packets/s'
' Flow IAT Mean'
' Flow IAT Std'
' Flow IAT Max'
' Flow IAT Min'
'Fwd IAT Total'
' Fwd IAT Mean'
' Fwd IAT Std'
' Fwd IAT Max'
' Fwd IAT Min'
'Bwd IAT Total'
' Bwd IAT Mean'
' Bwd IAT Std'
' Bwd IAT Max'
' Bwd IAT Min'
'Fwd PSH Flags'
' Bwd PSH Flags'
' Fwd URG Flags'
' Bwd URG Flags'
' Fwd Header Length'
' Bwd Header Length'
'Fwd Packets/s'
' Bwd Packets/s'
' Min Packet Length'
' Max Packet Length'
' Packet Length Mean'
' Packet Length Std'
' Packet Length Variance'
'FIN Flag Count'
' SYN Flag Count'
' RST Flag Count'
' PSH Flag Count'
' ACK Flag Count'
' URG Flag Count'
' CWE Flag 

In [17]:
import numpy as np

# Isolate numeric columns to compute descriptive statistics.
# The label column is a string in raw files, so filtering to
# numeric-only avoids errors and keeps the summary relevant.
numeric_sample = sample_df.select_dtypes(include=[np.number])

print("Numeric columns:", numeric_sample.shape[1])
print()
# describe() gives a quick view of ranges, means, and potential outliers
# across all numeric features in one call.
display(numeric_sample.describe())


Numeric columns: 78



,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,225745.00000,2.257450e+05,225745.000000,225745.000000,225745.000000,2.257450e+05,225745.000000,225745.000000,225745.000000,225745.000000,...,225745.000000,225745.000000,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05,2.257450e+05
mean,8879.61946,1.624165e+07,4.874916,4.572775,939.463346,5.960477e+03,538.535693,27.882221,164.826715,214.907242,...,3.311497,21.482753,1.848261e+05,1.293436e+04,2.080849e+05,1.776201e+05,1.032214e+07,3.611943e+06,1.287813e+07,7.755355e+06
std,19754.64740,3.152437e+07,15.422874,21.755356,3249.403484,3.921834e+04,1864.128991,163.324159,504.892965,797.411073,...,12.270018,4.166799,7.979250e+05,2.102737e+05,9.002350e+05,7.842602e+05,2.185303e+07,1.275689e+07,2.692126e+07,1.983109e+07
min,0.00000,-1.000000e+00,1.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,80.00000,7.118000e+04,2.000000,1.000000,26.000000,0.000000e+00,6.000000,0.000000,6.000000,0.000000,...,1.000000,20.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,80.00000,1.452333e+06,3.000000,4.000000,30.000000,1.640000e+02,20.000000,0.000000,8.666667,5.301991,...,2.000000,20.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,80.00000,8.805237e+06,5.000000,5.000000,63.000000,1.160100e+04,34.000000,6.000000,32.000000,10.263203,...,4.000000,20.000000,1.878000e+03,0.000000e+00,1.878000e+03,1.862000e+03,8.239725e+06,0.000000e+00,8.253838e+06,7.422849e+06
max,65532.00000,1.199999e+08,1932.000000,2942.000000,183012.000000,5.172346e+06,11680.000000,1472.000000,3867.000000,6692.644993,...,1931.000000,52.000000,1.000000e+08,3.950000e+07,1.000000e+08,1.000000e+08,1.200000e+08,6.530000e+07,1.200000e+08,1.200000e+08


In [18]:
# Re-check label distribution after verifying the column name format.
# This confirms which attack types appear in the raw data before
# the ETL pipeline collapses them to a binary label.
if " Label" in sample_df.columns:
    print("Label distribution in raw sample file:")
    print(sample_df[" Label"].value_counts())
elif "Label" in sample_df.columns:
    print("Label distribution in raw sample file:")
    print(sample_df["Label"].value_counts())
else:
    print("Label column not found.")


Label distribution in raw sample file:
DDoS      128027
BENIGN     97718
Name:  Label, dtype: int64
